In [9]:
select top 5  *
from ecommerce_raw;

(5 rows affected)

order_id         | order_date       | ship_date    | delivery_date    | customer_id | customer_name     | customer_email                | customer_segment | sales_channel | country        | region | product_id  | product_name              | category | subcategory | quantity | unit_price | unit_cogs | discount_pct | gross_revenue | net_revenue | cogs_total | gross_profit | gross_margin_pct | shipping_cost | return_flag | return_reason | return_date | refund_amount | inventory_on_hand | reorder_point | days_in_stock | supplier_id | supplier_lead_time_days | payment_method | payment_status | fulfillment_status | order_status | warehouse_id | notes         
-----------------+------------------+--------------+------------------+-------------+-------------------+-------------------------------+------------------+---------------+----------------+--------+-------------+---------------------------+----------+-------------+----------+------------+-----------+--------------+---

In [20]:
Select category, 
        subcategory,
        sum(round(net_revenue,0)) as Revenue,
        sum(round(gross_profit,0)) as Profit, 
        round(AVG(gross_margin_pct),2) AS Margin, 
        count(order_id) AS total_orders,
        (sum(net_revenue) * 100) / SUM(sum(net_revenue)) OVER() As Revenue_percentage

from FactSales
LEFT JOIN DimProduct on FactSales.product_id = DimProduct.product_id
GROUP BY category,subcategory
order by Revenue desc

(15 rows affected)

category | subcategory | Revenue  | Profit  | Margin   | total_orders | Revenue_percentage
---------+-------------+----------+---------+----------+--------------+-------------------
Office   | Desks       | 10003592 | 5284225 | 0.550000 | 7197         | 10.104012045259553
office   | Accessories | 8881881  | 4214574 | 0.470000 | 7041         | 8.971113699455987 
Bedroom  | Bedding     | 8193839  | 4624626 | 0.570000 | 7212         | 8.276277399049258 
Storage  | Boxes       | 7246421  | 4009181 | 0.510000 | 7192         | 7.319020779542074 
Kitchen  | Cookware    | 7200424  | 4266302 | 0.600000 | 7135         | 7.27272070259734  
Kitchen  | Cutlery     | 7146890  | 4025545 | 0.540000 | 7298         | 7.218571420257797 
Kitchen  | Utensils    | 6974451  | 3699697 | 0.550000 | 6047         | 7.04521165993137  
Storage  | Shelves     | 6852506  | 3689289 | 0.540000 | 5920         | 6.92102751123534  
Outdoor  | Patio       | 6546161  | 3897589 | 0.590000 | 5913         

Office leads revenue and profit, but kitchen and outdoor leads margin by 56% & 58% 

Office Accessories has the worst margin in the entire dataset at 47%, dragging the category average down. Office Desks at 55% and Chairs at 58% are healthy, but Accessories is quietly eating profit. Office leads revenue by a wide margin yet sits at 53% category margin precisely because of this subcategory. That's a concrete, actionable finding.

Cookware (60%), Cutlery (54%), Utensils (55%) — the tightest margin band of any category. No subcategory is underperforming. This is why Kitchen is the real star despite not leading revenue.

Bedding sits at 57%, Decor at 57%, but Lighting drops to 48%. Nearly 10 percentage points below its siblings. That's not a small gap — something structural is going on there, either pricing, COGS, or heavy discounting.

Boxes (51%) vs Shelves (54%) vs Baskets (54%) — margins are fine, but Baskets only generates $2.9M vs Boxes at $7.2M. Storage is heavily concentrated in Boxes. If that subcategory has a supply or demand issue, the whole category feels it.

Same margin as Patio (58-59%) but only $3.1M revenue vs $6.5M for Patio and $6M for Lighting. Less than half the orders of the other subcategories. Either it's a niche product line or it's underexposed in the catalog.



In [33]:
SELECT
    category,
    subcategory,
    ROUND(AVG(discount_pct), 3) AS avg_discount,
    ROUND(AVG(gross_margin_pct), 2) AS avg_margin,
     COUNT(FactReturns.order_id) * 100
        / COUNT(FactSales.order_id) AS return_rate_pct
FROM FactSales
LEFT JOIN DimProduct ON FactSales.product_id = DimProduct.product_id
LEFT JOIN FactReturns ON FactSales.order_id = FactReturns.order_id
GROUP BY category, subcategory
ORDER BY avg_discount DESC

(15 rows affected)

category | subcategory | avg_discount | avg_margin | return_rate_pct
---------+-------------+--------------+------------+----------------
Office   | Accessories | 0.078000     | 0.470000   | 26             
Outdoor  | Patio       | 0.078000     | 0.590000   | 28             
Office   | Desks       | 0.078000     | 0.550000   | 27             
Office   | Chairs      | 0.078000     | 0.580000   | 27             
Outdoor  | Gardening   | 0.077000     | 0.580000   | 26             
Storage  | Shelves     | 0.076000     | 0.540000   | 26             
Bedroom  | Lighting    | 0.076000     | 0.480000   | 27             
Bedroom  | Decor       | 0.076000     | 0.570000   | 27             
Kitchen  | Cutlery     | 0.076000     | 0.540000   | 27             
Storage  | Baskets     | 0.075000     | 0.540000   | 27             
Storage  | Boxes       | 0.075000     | 0.510000   | 33             
Kitchen  | Utensils    | 0.075000     | 0.550000   | 27             
Outdoor  | Lig

discounting is consistent across categories, so margin variation is not driven by promotional pricing
* Storage Boxes at 33% return rate but everything else is clustering between 26–28% return rate


In [86]:
SELECT sales_channel,AVG(net_revenue) as AOV,
         (( AVG(net_revenue) - LAG(AVG(net_revenue)) OVER (ORDER BY AVG(net_revenue) DESC))
    / LAG(AVG(net_revenue)) OVER (ORDER BY AVG(net_revenue) DESC)) * 100 AS pct_difference
from FactSales
GROUP BY sales_channel
order by AOV desc

(3 rows affected)

sales_channel | AOV               | pct_difference     
--------------+-------------------+--------------------
Wholesale     | 4127.604699627849 | NULL               
Amazon        | 686.9632875416057 | -83.35685373157116 
Shopify       | 663.2274292397035 | -3.4551858494278958
(3 rows)

Total execution time: 00:00:01.870

wholesale has an 83 percent higher than amazon, and 86 percent higher than shopify in the sales channels

In [85]:
SELECT customer_segment,AVG(net_revenue) as AOV,
    (( AVG(net_revenue) - LAG(AVG(net_revenue)) OVER (ORDER BY AVG(net_revenue) DESC))
     / LAG(AVG(net_revenue)) OVER (ORDER BY AVG(net_revenue) DESC)) * 100 AS pct_difference
from FactSales
LEFT JOIN DimCustomer on FactSales.customer_id = DimCustomer.customer_id
GROUP BY customer_segment


(3 rows affected)

customer_segment | AOV               | pct_difference     
-----------------+-------------------+--------------------
B2B              | 2639.177689584267 | NULL               
Wholesale        | 2594.528898313636 | -1.6917690478682368
B2C              | 624.453122869859  | -75.93192647514027 
(3 rows)

Total execution time: 00:00:04.804

b2b  has 1 percent higher aov than wholesale, and 76 percent higher than b2c

In [103]:
SELECT YEAR(order_date) AS YEAR,DATEPART(qq,order_date) AS QUARTER 
FROM FactSales
GROUP BY DATEPART(qq,order_date),YEAR(order_date)
ORDER BY YEAR(order_date), QUARTER

Error: Language client is not ready yet

In [37]:
SELECT  Year(order_date) AS Year,sum(round(net_revenue,0)) as Revenue,
        sum(round(gross_profit,0)) as Profit, 
        round(AVG(gross_margin_pct),2) AS Margin, 
        count(order_id) AS total_orders
FROM FactSales
GROUP BY Year(order_date)


(5 rows affected)

Year | Revenue  | Profit   | Margin   | total_orders
-----+----------+----------+----------+-------------
2022 | 32209138 | 17652075 | 0.550000 | 32904       
2023 | 33359556 | 18228346 | 0.550000 | 32915       
2024 | 32916805 | 18094543 | 0.550000 | 32744       
2025 | 81       | 39       | 0.480000 | 1           
2027 | 519340   | 278346   | 0.550000 | 489         
(5 rows)

Total execution time: 00:00:02.379

In [ ]:
SELECT top 20 order_date,ship_date, FactSales.order_id,return_date
FROM FactSales
LEFT JOIN FactReturns on FactSales.order_id = FactReturns.order_id
Where YEAR(order_date) = 2027 ;

In [53]:
WITH cleaned_data AS (SELECT order_id,
        CASE 
          WHEN YEAR(TRY_CAST (order_date AS DATE)) > YEAR(TRY_CAST (ship_date AS DATE)) 
                THEN REPLACE(order_date,  YEAR(order_date),YEAR(ship_date))
          
          else order_date
          end as order_date_cleaned, ship_date
from ecommerce_raw),

order_date_normalized as (
      SELECT 
      order_id,
      COALESCE(
            TRY_CAST(TRY_CONVERT(DATETIME,order_date_cleaned,101) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,order_date_cleaned,103) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,order_date_cleaned,105) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,order_date_cleaned,120) AS DATE)
            )AS order_date_1,
      
      COALESCE(
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,101) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,103) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,105) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,120) AS DATE)

            )AS ship_date
       
      from cleaned_data)

SELECT eccomerce_raw.order_id, order_date_1, order_date
from order_date_normalized
LEFT JOIN eccomerce_raw on order_date_normalized.order_id = eccomerce_raw.order_id
order by order_date_1 DESC

Msg 208, Level 16, State 1, Line 1
Invalid object name 'eccomerce_raw'.

Total execution time: 00:00:00.164

In [ ]:
WITH order_date_normalized AS 
 (SELECT order_id,
        CASE 
          WHEN YEAR(TRY_CAST (order_date AS DATE)) > YEAR(TRY_CAST (ship_date AS DATE)) 
                THEN REPLACE(order_date,  order_date,ship_date)
          
          else order_date
          end as _date, ship_date
from ecommerce_raw),

order_date_cleaned as (
      SELECT 
      order_id,
      COALESCE(
            TRY_CAST(TRY_CONVERT(DATETIME,_date,101) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,_date,103) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,_date,105) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,_date,120) AS DATE)
            )AS order_date,
      
      COALESCE(
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,101) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,103) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,105) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,120) AS DATE)

            )AS ship_date
       
      from order_date_normalized)



SELECT top 20 order_date_cleaned.order_date, ecommerce_raw.order_date,ecommerce_raw.ship_date
from order_date_cleaned
LEFT JOIN ecommerce_raw on order_date_cleaned.order_id = ecommerce_raw.order_id
order by order_date_cleaned.order_date DESC
;


In [73]:
WITH dates_cleaned AS (
    -- Step 1: Parse the string dates into actual DATE formats first
    SELECT 
        order_id,
        COALESCE(
            TRY_CAST(TRY_CONVERT(DATETIME, order_date, 101) AS DATE),
            TRY_CAST(TRY_CONVERT(DATETIME, order_date, 103) AS DATE),
            TRY_CAST(TRY_CONVERT(DATETIME, order_date, 105) AS DATE),
            TRY_CAST(TRY_CONVERT(DATETIME, order_date, 120) AS DATE)
        ) AS parsed_order_date,
        
        COALESCE(
            TRY_CAST(TRY_CONVERT(DATETIME, ship_date, 101) AS DATE),
            TRY_CAST(TRY_CONVERT(DATETIME, ship_date, 103) AS DATE),
            TRY_CAST(TRY_CONVERT(DATETIME, ship_date, 105) AS DATE),
            TRY_CAST(TRY_CONVERT(DATETIME, ship_date, 120) AS DATE)
        ) AS parsed_ship_date,
        order_date AS raw_order_date,
        ship_date AS raw_ship_date
    FROM ecommerce_raw
),

dates_normalized AS (
    SELECT 
        order_id,
        CASE 
            WHEN YEAR(parsed_order_date) > YEAR(parsed_ship_date) 
            THEN DATEFROMPARTS(
                YEAR(parsed_ship_date),
                MONTH(parsed_order_date),
                day(parsed_order_date) 
                )
            ELSE parsed_order_date 
        END AS final_order_date,
        parsed_ship_date AS final_ship_date,
        raw_order_date,
        raw_ship_date
    FROM dates_cleaned
)

-- Step 3: Final Selection
SELECT 
    final_order_date AS order_date, 
    raw_order_date, 
    raw_ship_date, ecommerce_raw.order_date
FROM dates_normalized
LEFT JOIN ecommerce_raw on dates_normalized.order_id = ecommerce_raw.order_id
WHERE ecommerce_raw.order_date LIKE '%2027%'

(1195 rows affected)

order_date | raw_order_date    | raw_ship_date     | order_date
-----------+-------------------+-------------------+-----------
2023-01-30 | 2027-01-30        | 2023-04-10        | 2027-01-30
2022-11-14 | 2022-11-14        | 2022-11-16        | 2027-02-10
2022-02-14 | 2027-02-14        | 11/26/2022        | 2027-02-14
2023-04-09 | 04/09/2023        | 11-04-2023        | 2027-03-06
2024-08-20 | 2024-08-20        | August 21 2024    | 2027-01-15
2022-10-26 | October 26 2022   | 28-10-2022        | 2027-01-12
2024-01-25 | 2027-01-25        | 29-02-2024        | 2027-01-25
2022-03-07 | 2027-03-07        | 2022-12-29        | 2027-03-07
2022-11-11 | November 11 2022  | 2022-11-14        | 2027-02-08
2022-09-21 | 2022-09-21        | 2022-09-24        | 2027-03-14
2023-03-16 | 2027-03-16        | 10-04-2023        | 2027-03-16
2024-03-05 | 2027-03-05        | 2024-04-17        | 2027-03-05
2023-03-19 | 2023-03-19        | 03/20/2023        | 2027-02-22
2023-07-15 | 2023-

In [6]:
SELECT COALESCE(
    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 101)) < YEAR(TRY_CONVERT(DATETIME, order_date, 101))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 101) AS DATE),
    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 101)) < YEAR(TRY_CONVERT(DATETIME, order_date, 101))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 101) AS DATE),
    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 105)) < YEAR(TRY_CONVERT(DATETIME, order_date, 105))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 105) AS DATE),

    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 120)) < YEAR(TRY_CONVERT(DATETIME, order_date, 120))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 103)  AS DATE) 
) AS order_date


FROM ecommerce_raw
WHERE order_date LIKE '%2027%'



(493 rows affected)

order_date
----------
2023-01-30
2022-02-14
2027-01-25
2022-03-07
2023-03-16
2024-03-05
2023-03-27
2023-03-01
2022-01-02
2024-01-07
2023-03-06
2024-03-09
2024-02-01
2027-04-07
2022-01-11
2023-03-08
2023-03-27
2022-04-07
2024-03-30
2024-02-05
2027-01-15
2024-03-23
2022-02-14
2027-03-14
2027-02-21
2023-02-27
2023-02-10
2023-02-07
2024-01-02
2027-02-17
2024-01-10
2022-02-22
2023-02-06
2024-03-05
2023-03-12
2023-02-20
2022-04-07
2023-01-14
2022-03-06
2022-01-16
2022-01-20
2023-01-28
2024-03-30
2024-03-31
2023-03-08
2023-04-10
2024-03-26
2024-01-31
2024-03-13
2022-01-20
2024-01-20
2022-04-04
2025-04-05
2024-03-28
2022-01-15
2023-01-18
2022-01-05
2027-03-13
2024-03-22
2023-03-10
2024-03-03
2023-03-08
2024-01-11
2023-02-04
2027-01-09
2024-01-14
2022-02-11
2022-01-17
2022-01-07
2022-02-16
2024-03-07
2024-02-11
2024-03-18
2023-01-21
2024-01-12
2024-03-26
2027-03-01
2023-02-02
2022-01-25
2022-03-15
2023-01-25
2024-03-03
2023-02-01
2022-03-21
2024-02-06
2024-02-05
2023-04-09
